# Descriptive Statistics — AI Engineering Interview Prep

Covers: central tendency, spread, shape, outliers, correlation, visualization.

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use("seaborn-v0_8")

# Generate datasets
income = np.random.lognormal(mean=10.5, sigma=0.8, size=500)   # right-skewed
exam_scores = np.random.normal(loc=72, scale=12, size=300).clip(0, 100)
age = np.random.randint(18, 70, size=200).astype(float)

print(f"Income:  n={len(income)}, dtype={income.dtype}")
print(f"Scores:  n={len(exam_scores)}, range=[{exam_scores.min():.1f}, {exam_scores.max():.1f}]")

## 1. Central Tendency

In [ ]:
for name, data in [("income", income), ("exam_scores", exam_scores)]:
    mean   = np.mean(data)
    median = np.median(data)
    mode_r = stats.mode(data, keepdims=True)
    print(f"\n{name}:")
    print(f"  mean   = {mean:>10.2f}")
    print(f"  median = {median:>10.2f}")
    print(f"  mode   = {mode_r.mode[0]:>10.2f}  (count={mode_r.count[0]})")
    print(f"  skew tells: mean > median when right-skewed ({mean > median})")

## 2. Spread (Variability)

In [ ]:
data = exam_scores
q1, q3 = np.percentile(data, [25, 75])
iqr = q3 - q1

print(f"Range:  {data.max() - data.min():.2f}")
print(f"Variance: {np.var(data, ddof=1):.2f}  (sample, ddof=1)")
print(f"Std dev:  {np.std(data, ddof=1):.2f}")
print(f"Q1={q1:.2f}, Q3={q3:.2f}, IQR={iqr:.2f}")
print(f"CV (coeff of variation): {np.std(data)/np.mean(data)*100:.1f}%")
print(f"\nFive-number summary:")
print(f"  min={data.min():.2f}, Q1={q1:.2f}, median={np.median(data):.2f}, Q3={q3:.2f}, max={data.max():.2f}")

## 3. Shape (Skewness & Kurtosis)

In [ ]:
for name, data in [("income (skewed)", income), ("scores (normal)", exam_scores)]:
    sk  = stats.skew(data)
    kurt = stats.kurtosis(data)  # excess kurtosis (normal=0)
    print(f"{name}:")
    print(f"  skewness = {sk:.3f}  (>0 right-skewed, <0 left-skewed)")
    print(f"  kurtosis = {kurt:.3f} (excess; >0 heavy-tailed 'leptokurtic')")
    print()

## 4. Outlier Detection

In [ ]:
data = exam_scores
q1, q3 = np.percentile(data, [25, 75])
iqr = q3 - q1

# IQR method
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outliers_iqr = data[(data < lower_fence) | (data > upper_fence)]
print(f"IQR fences: [{lower_fence:.2f}, {upper_fence:.2f}]")
print(f"IQR outliers: {len(outliers_iqr)} ({len(outliers_iqr)/len(data)*100:.1f}%)")

# Z-score method
z_scores = np.abs((data - data.mean()) / data.std())
outliers_z = data[z_scores > 3]
print(f"Z-score outliers (|z|>3): {len(outliers_z)}")

## 5. Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Histogram + KDE
axes[0,0].hist(exam_scores, bins=20, density=True, alpha=0.6, color="steelblue", label="scores")
x = np.linspace(exam_scores.min(), exam_scores.max(), 200)
axes[0,0].plot(x, stats.norm.pdf(x, exam_scores.mean(), exam_scores.std()), "r-", lw=2, label="normal fit")
axes[0,0].set_title("Histogram + Normal Fit")
axes[0,0].legend()

# Box plot
axes[0,1].boxplot([exam_scores, income/income.mean()*50], labels=["Scores", "Income (scaled)"])
axes[0,1].set_title("Box Plot Comparison")

# Violin plot
axes[1,0].violinplot([exam_scores], positions=[1], showmedians=True)
axes[1,0].set_title("Violin Plot (Exam Scores)")
axes[1,0].set_xticks([1])
axes[1,0].set_xticklabels(["Scores"])

# QQ plot
stats.probplot(exam_scores, dist="norm", plot=axes[1,1])
axes[1,1].set_title("QQ Plot")

plt.tight_layout()
plt.show()

## 6. Correlation

In [ ]:
study_hours = np.random.uniform(1, 10, 100)
test_scores = 40 + 5 * study_hours + np.random.normal(0, 5, 100)

pearson_r, pearson_p = stats.pearsonr(study_hours, test_scores)
spearman_r, spearman_p = stats.spearmanr(study_hours, test_scores)

print(f"Pearson r  = {pearson_r:.3f}  (p={pearson_p:.4f}) — linear correlation")
print(f"Spearman r = {spearman_r:.3f}  (p={spearman_p:.4f}) — rank-based (robust to outliers)")

plt.figure(figsize=(6, 4))
plt.scatter(study_hours, test_scores, alpha=0.5)
m, b = np.polyfit(study_hours, test_scores, 1)
plt.plot(sorted(study_hours), [m*x+b for x in sorted(study_hours)], "r-")
plt.xlabel("Study Hours")
plt.ylabel("Test Score")
plt.title(f"Correlation: Pearson r={pearson_r:.2f}")
plt.show()

## Interview Tips

- **Mean vs Median**: median is robust to outliers; for skewed data (salaries, prices) prefer median.
- **ddof=1** for sample std; **ddof=0** for population. Most real-world: sample (ddof=1).
- **IQR method** for outliers is rule-of-thumb. **Z-score** assumes normality.
- **Kurtosis**: positive = heavy tails (more extreme values than normal); negative = light tails.
- **Spearman** doesn't assume linearity — better for monotonic but non-linear relationships.
- QQ plot: points along diagonal = normally distributed; S-curve = skewed.